In [ ]:
# chain --> to crete pipeline ( output of on become input for others )
# typeof chain --> sequential , parallel , conditional

In [5]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from typing import TypedDict , Annotated , List , Optional
from datetime import datetime
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser , JsonOutputParser 

load_dotenv()
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

llm_gemini = ChatGoogleGenerativeAI(model="gemini-2.0-flash" , api_key= GOOGLE_API_KEY)
llm_gemini.invoke("who is father of india").content

'Mahatma Gandhi is widely considered the "Father of India."'

In [9]:
# Sequntial chain
template1 = PromptTemplate(
    template= "write a deatailed report on the {topic}",
    input_variables= ['topic']
    )

template2 = PromptTemplate(
    template= "Give me 5 point summary for this given text: \n {text}",
    input_variables= ['text']
    )

parser = StrOutputParser()

chain = template1 | llm_gemini | parser | template2 | llm_gemini | parser

print(chain.invoke("Data Scientist Field"))
# print(chain.get_graph().print_ascii())

Here's a 5-point summary of the provided text:

*   **Data science is an interdisciplinary field combining statistics, computer science, and domain expertise to extract knowledge and insights from data for data-informed decision-making.** It differs from data analysis (focused on past events) and data engineering (focused on data infrastructure).
*   **Successful data scientists require a diverse skill set, including technical skills (programming, statistics, machine learning, database management), analytical skills (problem-solving, critical thinking), and soft skills (communication, teamwork, business acumen).**
*   **Data scientist responsibilities vary but commonly include data collection and preparation, data exploration and analysis, model building and evaluation, model deployment and monitoring, data visualization and reporting, and communication with stakeholders.**
*   **The data science field offers various career paths (e.g., data scientist, machine learning engineer, data e

In [11]:
# Building parallel chains
from langchain.schema.runnable import RunnableParallel

parser = StrOutputParser()

template1 = PromptTemplate(
    template="give me short and sharp notes for the txt i'm giving you \n {text}",
    input_variables=['text']
)
template2 = PromptTemplate(
    template= "give me 5 question with answer from thr following text \n {text}",
    input_variables= ['text']
)
template3 = PromptTemplate(
    template= 'combine the provide notes and document in the same document \n notes : {notes} \n quiz : {quiz}',
    input_variables=['notes' , 'quiz']
)

parallel_chain = RunnableParallel({
    "notes" : template1 | llm_gemini | parser,
    "quiz" : template2 | llm_gemini | parser
})

final_chain = parallel_chain | template3 | llm_gemini | parser

text = """
Qwen3-480B-A35B-Instruct has the following features:

Type: Causal Language Models
Training Stage: Pretraining & Post-training
Number of Parameters: 480B in total and 35B activated
Number of Layers: 62
Number of Attention Heads (GQA): 96 for Q and 8 for KV
Number of Experts: 160
Number of Activated Experts: 8
Context Length: 262,144 natively.
NOTE: This model supports only non-thinking mode and does not generate <think></think> blocks in its output. Meanwhile, specifying enable_thinking=False is no longer required.

For more details, including benchmark evaluation, hardware requirements, and inference performance, please refer to our blog, GitHub, and Documentation.

Quickstart
We advise you to use the latest version of transformers.

With transformers<4.51.0, you will encounter the following error:

KeyError: 'qwen3_moe'
"""
print(final_chain.invoke(text))

# Qwen3-480B-A35B-Instruct Model Details and Quiz

This document combines technical notes and a quiz about the Qwen3-480B-A35B-Instruct Causal Language Model.

## Model Details:

*   **Model:** Qwen3-480B-A35B-Instruct
*   **Type:** Causal Language Model
*   **Training:** Pretrained & Post-trained
*   **Parameters:** 480B (total), 35B (activated)
*   **Layers:** 62
*   **Attention Heads:** 96 (Q), 8 (KV) - GQA
*   **Experts:** 160 total, 8 activated
*   **Context Length:** 262,144
*   **Non-Thinking Mode Only:** No `<think>` blocks. `enable_thinking=False` not needed.
*   **Details:** See blog, GitHub, Documentation.
*   **Quickstart:** Use latest `transformers` version.
*   **Error if transformers < 4.51.0:** `KeyError: 'qwen3_moe'`

## Quiz:

Okay, here are five questions with answers based on the provided text about Qwen3-480B-A35B-Instruct:

**Question 1:** What are the two training stages for the Qwen3-480B-A35B-Instruct model?
**Answer:** Pretraining & Post-training

**Question 2

In [12]:
final_chain.get_graph().print_ascii()

                    +---------------------------+                      
                    | Parallel<notes,quiz>Input |                      
                    +---------------------------+                      
                       ***                   ***                       
                   ****                         ****                   
                 **                                 **                 
    +----------------+                          +----------------+     
    | PromptTemplate |                          | PromptTemplate |     
    +----------------+                          +----------------+     
             *                                           *             
             *                                           *             
             *                                           *             
+------------------------+                  +------------------------+ 
| ChatGoogleGenerativeAI |                  | ChatGoogleGenerati

In [ ]:
# Building Conditional chains
from pydantic import BaseModel , Field
from typing import Literal
from langchain_core.output_parsers import PydanticOutputParser
from langchain.schema.runnable import RunnableBranch

class Classifiction(BaseModel):
    sentiment : Literal['Postitive' , 'Negative'] = Field(description= "sentiment of the given text")

pydantic_parser = PydanticOutputParser(pydantic_object=Classifiction)

template1 = PromptTemplate(
    template= "Classify the follwoing text into a nagative or positive sentiment \n {text} and {format_intstr}",
    input_variables= ['text'],
    partial_variables= {'format_intstr' : pydantic_parser.get_format_instructions()}
)

template2 = PromptTemplate(
    template= "write me an appopirate Resnponse to this postive text : \n {text}",
    input_variables= 
)

classifier_chain = template1 | llm_gemini | pydantic_parser

final_chain = RunnableBranch()
print(classifier_chain.invoke("this is a Very Good mobile pphone"))

sentiment='Postitive'
